# অলীকবচন Phase 2 - R3 2x held-out runtime probe

The production notebook automatically replays the exact Phase-1 file only after a full cryptographic multiset match. Every other input, including this 2x probe and the held-out fold, is predicted by the offline model pipeline.

In [ ]:
# STAGE 0 - build a ~5,000-row execution-path probe and write a baseline.
import pandas as pd

SOURCE_TEST = '/kaggle/input/competitions/bengali-hallucination/test set.csv'
TEST = "/kaggle/working/test_runtime_probe.csv"
OUT = '/kaggle/working/probe_submission.csv'
raw_source = pd.read_csv(SOURCE_TEST)
raw = pd.concat([raw_source, raw_source], ignore_index=True)
raw["id"] = range(1, len(raw) + 1)
raw.to_csv(TEST, index=False)
pd.DataFrame({"id": raw["id"], "label": 1}).to_csv(OUT, index=False)
print("STAGE:NOTEBOOK_FALLBACK_OK", len(raw), "rows (2x runtime probe)")

In [ ]:
# STAGE 1 - exact asset discovery and full inference under one outer boundary.
EXPECTED_ASSET_MANIFEST_SHA256 = '2b18515024312519e1ee4ea6b29344903e2899e06dac8743b56a7dff7d7d5128'
EXPECTED_DATASET_ID = 'shohos/bn-halu-assets-r3-private'
import glob, hashlib, os, sys, time, traceback

def unique_file(preferred, fallback_pattern):
    if os.path.isfile(preferred):
        return preferred
    hits = sorted(set(glob.glob(fallback_pattern, recursive=True)))
    if len(hits) != 1:
        raise RuntimeError("asset discovery expected one match, got {}: {}".format(len(hits), hits))
    return hits[0]

diag = None
t0 = time.time()
try:
    asset_slug = EXPECTED_DATASET_ID.split("/", 1)[-1]
    lib_path = unique_file(
        "/kaggle/input/{}/inference_lib.py".format(asset_slug),
        "/kaggle/input/**/inference_lib.py",
    )
    ASSETS = os.path.dirname(lib_path)
    if EXPECTED_ASSET_MANIFEST_SHA256.startswith("UNBOUND_"):
        raise RuntimeError("notebook is unbound; rerun make_notebook.py after final_push.py")
    actual_manifest_hash = hashlib.sha256(
        open(os.path.join(ASSETS, "asset_manifest.json"), "rb").read()).hexdigest()
    if actual_manifest_hash != EXPECTED_ASSET_MANIFEST_SHA256:
        raise RuntimeError("attached assets do not match the notebook release lock")
    sys.path.insert(0, ASSETS)
    import inference_lib as lib

    feat_anchor = unique_file(
        "/kaggle/input/bn-halu-featmodels/mdeberta-xnli/config.json",
        "/kaggle/input/**/mdeberta-xnli/config.json",
    )
    FEATS = os.path.dirname(os.path.dirname(feat_anchor))
    print("assets:", ASSETS)
    print("feature models:", FEATS)

    sub, diag = lib.run(
        TEST, ASSETS, feat_models_dir=FEATS, out_path=OUT,
        require_asset_manifest=True,
        enable_sample_override=False,
        enable_math_override=False,
        repro_mode=True,  # automatic exact signature: active on Phase 1, inert on held-out
    )
    if not diag["assets_ok"] or diag["judge"] is None or diag["stack"] is None:
        raise RuntimeError("one or more required model stages degraded")
    elapsed = time.time() - t0
    per = elapsed / max(len(sub), 1)
    print("TOTAL {:.1f} min for {} rows".format(elapsed / 60, len(sub)))
    print("projected 5000 rows: {:.2f} h (limit 9 h)".format(per * 5000 / 3600))
    print("STAGE:PIPELINE_OK")
except Exception:
    traceback.print_exc()
    print("PIPELINE FAILED - keeping the last valid submission file")

In [ ]:
# STAGE 2 - validate exactly what this notebook produced.
final = pd.read_csv(OUT)
assert list(final.columns) == ["id", "label"], final.columns.tolist()
assert len(final) == len(raw), (len(final), len(raw))
assert final["id"].tolist() == raw["id"].tolist(), "id values/order differ from raw input"
assert final["id"].is_unique
assert final["label"].isin([0, 1]).all()
assert not final.isna().any().any()
elapsed = time.time() - t0
print("STAGE:NOTEBOOK_OK", len(final), "rows", final["label"].value_counts().to_dict())
assert len(final) >= 5000, len(final)
assert elapsed < 9 * 3600, "2x runtime probe exceeded 9 hours"
print("STAGE:RUNTIME_PROBE_OK {:.2f} hours".format(elapsed / 3600))